In [1]:
# ========================================================
# 📘 VisoSpeak - 02_feature_extraction.ipynb
# ========================================================
"""
This notebook extracts visual features from the cropped lip regions using:
- 3D-CNN (r3d_18) → for temporal patterns
- 2D-ResNet18 → for per-frame spatial features

🔹 INPUT:
    - Original video from: data/raw_videos/{video_name}.mpg
    - Mouth crop coords from: data/processed/{video_name}/crop_metadata_{video_name}.json

🔹 OUTPUT:
    - ResNet features: data/features/{video_name}/resnet_features.npy
    - R3D features: data/features/{video_name}/r3d_features.npy
    - Metadata log: data/features/{video_name}/feature_metadata_log.json
"""


'\nThis notebook extracts visual features from the cropped lip regions using:\n- 3D-CNN (r3d_18) → for temporal patterns\n- 2D-ResNet18 → for per-frame spatial features\n\n🔹 INPUT:\n    - Original video from: data/raw_videos/{video_name}.mpg\n    - Mouth crop coords from: data/processed/{video_name}/crop_metadata_{video_name}.json\n\n🔹 OUTPUT:\n    - ResNet features: data/features/{video_name}/resnet_features.npy\n    - R3D features: data/features/{video_name}/r3d_features.npy\n    - Metadata log: data/features/{video_name}/feature_metadata_log.json\n'

In [2]:
# ========================================================
# ✅ STEP 1: IMPORTS & SETUP
# ========================================================
import os
import json
import cv2
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms
import torchvision.models.video as video_models
import torchvision.models as img_models

DATA_DIR = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data"
video_name = "bbaf2n"
video_path = os.path.join(DATA_DIR, "raw_videos", f"{video_name}.mpg")
metadata_path = os.path.join(DATA_DIR, "processed", video_name, f"crop_metadata_{video_name}.json")
output_dir = os.path.join(DATA_DIR, "processed", video_name)
os.makedirs(output_dir, exist_ok=True)


In [3]:
# ========================================================
# ✅ STEP 2: LOAD PRETRAINED MODELS
# ========================================================
print("🔄 Loading pretrained models...")
r3d_full = video_models.r3d_18(pretrained=True)
r3d = torch.nn.Sequential(*list(r3d_full.children())[:-1])
r3d.eval()

resnet = img_models.resnet18(pretrained=True)
resnet.eval()
print("✅ Models loaded successfully.")


🔄 Loading pretrained models...


C:\Users\PC\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\PC\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\PC\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=Res

✅ Models loaded successfully.


In [5]:
# ========================================================
# ✅ STEP 3: DEFINE TRANSFORMS
# ========================================================
video_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.43216, 0.394666, 0.37645],
                         std=[0.22803, 0.22145, 0.216989])
])
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [6]:
# ========================================================
# ✅ STEP 4: LOAD METADATA & CROP FRAMES
# ========================================================
with open(metadata_path, "r") as f:
    metadata = json.load(f)

crop_coords = tuple(metadata["crop_coords"])
print("🎥 Video:", video_path)
print("📐 Crop coords:", crop_coords)

def extract_lip_frames(video_path, crop_coords):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        x, y, w, h = crop_coords
        crop = frame[y:y+h, x:x+w]
        img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        frames.append(img)

    cap.release()
    print(f"✅ Loaded {len(frames)} cropped frames.")
    return frames

lip_frames = extract_lip_frames(video_path, crop_coords)


🎥 Video: C:\Users\PC\Desktop\phaseB\VisoSpeak\data\raw_videos\bbaf2n.mpg
📐 Crop coords: (129, 206, 59, 35)
✅ Loaded 75 cropped frames.


In [7]:
# ========================================================
# ✅ STEP 5: FEATURE EXTRACTION FUNCTIONS
# ========================================================
def extract_resnet_features(frames):
    print("🧠 Running 2D-ResNet18...")
    features = []
    for frame in frames:
        tensor = image_transform(frame).unsqueeze(0)
        with torch.no_grad():
            feat = resnet(tensor).squeeze().numpy()
        features.append(feat)
    return np.array(features)

def extract_3dcnn_features(frames, clip_length=16):
    print("🧠 Running 3D-CNN (r3d_18)...")
    features = []
    for i in range(0, len(frames) - clip_length + 1, clip_length):
        clip = frames[i:i+clip_length]
        clip_tensor = torch.stack([video_transform(f) for f in clip])
        clip_tensor = clip_tensor.unsqueeze(0).permute(0, 2, 1, 3, 4)  # BCTHW
        with torch.no_grad():
            feat = r3d(clip_tensor).squeeze().numpy()
        features.append(feat)
    return np.array(features)


In [8]:
# ========================================================
# ✅ STEP 6: RUN EXTRACTION AND SAVE OUTPUT
# ========================================================
resnet_feats = extract_resnet_features(lip_frames)
resnet_path = os.path.join(output_dir, "resnet_features.npy")
np.save(resnet_path, resnet_feats)
print(f"💾 Saved ResNet features: {resnet_path}")

r3d_feats = extract_3dcnn_features(lip_frames)
r3d_path = os.path.join(output_dir, "r3d_features.npy")
np.save(r3d_path, r3d_feats)
print(f"💾 Saved R3D features: {r3d_path}")


🧠 Running 2D-ResNet18...
💾 Saved ResNet features: C:\Users\PC\Desktop\phaseB\VisoSpeak\data\processed\bbaf2n\resnet_features.npy
🧠 Running 3D-CNN (r3d_18)...
💾 Saved R3D features: C:\Users\PC\Desktop\phaseB\VisoSpeak\data\processed\bbaf2n\r3d_features.npy


In [2]:
import cv2
import os
import json

# Setup paths
video_path = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data\raw_videos\vid_001.mpg"
video_name = os.path.splitext(os.path.basename(video_path))[0]
video_dir = os.path.join(r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data", video_name)

frames_dir = os.path.join(video_dir, "frames")
mouth_crops_dir = os.path.join(video_dir, "mouth_crops")
metadata_path = os.path.join(video_dir, "metadata.json")

# Load Haar cascade for mouth detection
mouth_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_mcs_mouth.xml')

# Load metadata
with open(metadata_path, "r") as f:
    metadata = json.load(f)

for entry in metadata:
    frame_path = os.path.join(frames_dir, entry["frame"])
    img = cv2.imread(frame_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Detect mouths
    mouths = mouth_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    if len(mouths) > 0:
        # Use the largest detected region (usually the actual mouth)
        x, y, w, h = sorted(mouths, key=lambda r: r[2]*r[3], reverse=True)[0]
        mouth_crop = img[y:y+h, x:x+w]

        # Save the mouth crop
        crop_path = os.path.join(mouth_crops_dir, entry["frame"])
        cv2.imwrite(crop_path, mouth_crop)
    else:
        print(f"No mouth found in frame {entry['frame']}")

print(f"✅ Mouth crops saved to {mouth_crops_dir}")


✅ Mouth crops saved to C:\Users\PC\Desktop\phaseB\VisoSpeak\data\vid_001\mouth_crops


In [11]:
import os
import numpy as np

DATA_DIR = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data"
video_name = "bbaf2n"
feature_dir = os.path.join(DATA_DIR, "processed", video_name)

resnet_path = os.path.join(feature_dir, "resnet_features.npy")
r3d_path = os.path.join(feature_dir, "r3d_features.npy")

resnet = np.load(resnet_path)
r3d = np.load(r3d_path)

print("✅ ResNet shape:", resnet.shape)
print("✅ R3D shape:", r3d.shape)


✅ ResNet shape: (75, 1000)
✅ R3D shape: (4, 512)
